# Modular RF Cluster-Size Analysis

This notebook is the reproducible, modular version of `rf_cluster_size.ipynb`. Most logic lives in `cluster_size_rf/lib`; this notebook orchestrates the run: prepare data, fit models, save tables/models, and make diagnostic plots.

For sequence-level modelling, each sequence is assigned to one primary-resolution cluster across overlapping windows. The default assignment is `largest_cluster`: choose the largest cluster the sequence appears in, then break ties by closeness to the window midpoint and earlier window index. The legacy `closest_mid_date` rule remains available as a sensitivity option in `AnalysisConfig`.


## 1. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path('/Users/ydnkka/Desktop/PhD Project/projects/scotland')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

from cluster_size_rf.lib.config import AnalysisConfig, DATA_PATH, OUT_DIR
from cluster_size_rf.lib.data import (
    make_outcome_tables,
    prepare_model_inputs,
    read_analysis_data,
)
from cluster_size_rf.lib.models import (
    fit_binary_model,
    fit_multiclass_model,
    make_split_data,
    permutation_importance_table,
)
from cluster_size_rf.lib.outputs import (
    save_binary_result,
    save_multiclass_result,
    save_outcome_tables,
    save_run_metadata,
    save_sensitivity_metrics,
)
from cluster_size_rf.lib.plots import (
    accumulated_local_effects_curves,
    partial_dependence_curves,
    plot_ale_curves,
    plot_binary_diagnostics,
    plot_importance,
    plot_outcome_counts,
    plot_partial_dependence_curves,
)
from cluster_size_rf.lib.sensitivity import (
    run_resolution_sensitivity,
    run_threshold_sensitivity,
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data: {DATA_PATH}')
print(f'Outputs: {OUT_DIR}')

## 2. Configuration

Use `run_mode='quick'` while iterating and `run_mode='final'` for the main run. The main model uses one overall SIMD measure; the domain decomposition model replaces it with all seven SIMD domains using the selected rank/quintile/decile mode.

`window_dedup_strategy='largest_cluster'` means each sequence is kept once, assigned to the largest cluster it belongs to across overlapping windows. Use `closest_mid_date` only if you want the original midpoint-based sensitivity.


In [ ]:
config = AnalysisConfig.from_run_mode(
    run_mode='final',  # change to 'final' for the main analysis
    simd_overall_mode='quintile',  # options: 'quintile', 'decile', 'rank'
    simd_domain_mode='quintile',  # options: 'quintile', 'decile', 'rank'
    simd_band_weighting='population',  # options: 'population' (PHS-style), 'equal_rank'
    window_dedup_strategy='largest_cluster',  # options: 'largest_cluster', 'closest_mid_date'
    use_sensitivity_context_controls=True,
    fit_singleton_binary=True,
    fit_simd_domain_decomposition=True,
    fit_secondary_multiclass=True,
    run_threshold_sensitivity=True,
    run_resolution_sensitivity=True,
    save_models=True,
)
config

## 3. Prepare Data

This step filters to the primary clustering resolution, keeps good/mediocre QC sequences, and assigns one cluster per sequence using `config.window_dedup_strategy`.


In [ ]:
raw = read_analysis_data(DATA_PATH)
model_df, X, feature_spec, targets = prepare_model_inputs(raw, config)
outcome_tables = make_outcome_tables(model_df)

print(f'Raw rows: {len(raw):,}')
print(f'Modelling rows: {len(model_df):,}')
print(f'Unique sequences: {model_df["sequence_id"].nunique():,}')
print(f'Unique clusters: {model_df["cluster_id"].nunique():,}')
print(f'Window de-duplication strategy: {config.window_dedup_strategy}')
print(f'Main SIMD feature: {feature_spec.simd_overall_feature}')
print(f'Main feature matrix: {X.shape}')
print(f'Decomposition features: {len(feature_spec.decomposition_features)}')

display(outcome_tables['outcome_counts'])
display(outcome_tables['test_reason_counts'])
display(outcome_tables['unmapped_test_reasons'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_outcome_counts(outcome_tables['outcome_counts'], ax=ax)
plt.tight_layout()
fig.savefig(FIG_DIR / 'primary_outcome_counts.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG_DIR / 'primary_outcome_counts.pdf', bbox_inches='tight')
print(f'Saved outcome-count figure to {FIG_DIR}')

## 4. Grouped Holdout Split

In [ ]:
split = make_split_data(X, targets['large'], targets['groups'], config)
print(f'Splitter: {split.splitter_name}')
print(f'Train rows: {len(split.X_train):,}; train clusters: {split.groups_train.nunique():,}')
print(f'Test rows : {len(split.X_test):,}; test clusters : {split.groups_test.nunique():,}')
display(pd.concat([
    split.y_train.value_counts(normalize=True).rename('train'),
    split.y_test.value_counts(normalize=True).rename('test'),
], axis=1).rename(index={0: 'not_large', 1: 'large'}).round(3))

## 5. Primary Binary Model: Large Cluster vs Not Large

In [ ]:
large_result = fit_binary_model(
    name='binary_large_cluster',
    X_train=split.X_train,
    y_train=split.y_train,
    X_test=split.X_test,
    y_test=split.y_test,
    config=config,
    numeric_features=feature_spec.numeric_features,
    categorical_features=feature_spec.categorical_features,
    negative_name='not_large',
    positive_name='large',
)
large_result.importance = permutation_importance_table(
    large_result.model,
    split.X_test,
    split.y_test,
    feature_spec.all_features,
    config,
)

display(pd.DataFrame([large_result.metrics]).round(3))
display(large_result.confusion_matrix)
display(large_result.classification_report.round(3))
display(large_result.importance)

In [ ]:
large_diag_fig = plot_binary_diagnostics(large_result, split.y_test, 'Large cluster')
large_diag_fig.savefig(FIG_DIR / 'binary_large_cluster_diagnostics.png', dpi=300, bbox_inches='tight')
large_diag_fig.savefig(FIG_DIR / 'binary_large_cluster_diagnostics.pdf', bbox_inches='tight')

fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * min(25, len(large_result.importance)))))
plot_importance(large_result.importance, 'Large-cluster permutation importance', ax=ax)
fig.savefig(FIG_DIR / 'binary_large_cluster_permutation_importance.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG_DIR / 'binary_large_cluster_permutation_importance.pdf', bbox_inches='tight')
plt.show()
print(f'Saved large-cluster diagnostic figures to {FIG_DIR}')

## 6. Secondary Binary Model: Singleton vs Clustered

In [ ]:
singleton_result = None
if config.fit_singleton_binary:
    y_singleton_train = targets['singleton'].loc[split.X_train.index]
    y_singleton_test = targets['singleton'].loc[split.X_test.index]
    singleton_result = fit_binary_model(
        name='binary_singleton_clustered',
        X_train=split.X_train,
        y_train=y_singleton_train,
        X_test=split.X_test,
        y_test=y_singleton_test,
        config=config,
        numeric_features=feature_spec.numeric_features,
        categorical_features=feature_spec.categorical_features,
        negative_name='clustered',
        positive_name='singleton',
    )
    singleton_result.importance = permutation_importance_table(
        singleton_result.model,
        split.X_test,
        y_singleton_test,
        feature_spec.all_features,
        config,
    )
    display(pd.DataFrame([singleton_result.metrics]).round(3))
    display(singleton_result.confusion_matrix)
    display(singleton_result.importance)
if singleton_result is not None:
    singleton_diag_fig = plot_binary_diagnostics(singleton_result, y_singleton_test, 'Singleton')
    singleton_diag_fig.savefig(FIG_DIR / 'binary_singleton_clustered_diagnostics.png', dpi=300, bbox_inches='tight')
    singleton_diag_fig.savefig(FIG_DIR / 'binary_singleton_clustered_diagnostics.pdf', bbox_inches='tight')
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * min(25, len(singleton_result.importance)))))
    plot_importance(singleton_result.importance, 'Singleton-versus-clustered permutation importance', ax=ax, color='#66AA55')
    fig.savefig(FIG_DIR / 'binary_singleton_clustered_permutation_importance.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / 'binary_singleton_clustered_permutation_importance.pdf', bbox_inches='tight')
    plt.show()
    print(f'Saved singleton diagnostic figures to {FIG_DIR}')

## 7. SIMD Domain Decomposition

In [ ]:
simd_domain_result = None
if config.fit_simd_domain_decomposition:
    X_decomp = model_df[feature_spec.decomposition_features].copy()
    X_decomp_train = X_decomp.loc[split.X_train.index]
    X_decomp_test = X_decomp.loc[split.X_test.index]
    simd_domain_result = fit_binary_model(
        name='simd_domain_large_cluster',
        X_train=X_decomp_train,
        y_train=split.y_train,
        X_test=X_decomp_test,
        y_test=split.y_test,
        config=config,
        numeric_features=feature_spec.decomposition_numeric_features,
        categorical_features=feature_spec.categorical_features,
        negative_name='not_large',
        positive_name='large',
    )
    simd_domain_result.importance = permutation_importance_table(
        simd_domain_result.model,
        X_decomp_test,
        split.y_test,
        feature_spec.decomposition_features,
        config,
    )
    display(pd.DataFrame([simd_domain_result.metrics]).round(3))
    display(simd_domain_result.importance)
if simd_domain_result is not None:
    simd_domain_diag_fig = plot_binary_diagnostics(simd_domain_result, split.y_test, 'SIMD domains: large cluster')
    simd_domain_diag_fig.savefig(FIG_DIR / 'simd_domain_large_cluster_diagnostics.png', dpi=300, bbox_inches='tight')
    simd_domain_diag_fig.savefig(FIG_DIR / 'simd_domain_large_cluster_diagnostics.pdf', bbox_inches='tight')
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * min(30, len(simd_domain_result.importance)))))
    plot_importance(simd_domain_result.importance, 'SIMD-domain permutation importance', ax=ax, top_n=30, color='#AA7744')
    fig.savefig(FIG_DIR / 'simd_domain_large_cluster_permutation_importance.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / 'simd_domain_large_cluster_permutation_importance.pdf', bbox_inches='tight')
    plt.show()
    print(f'Saved SIMD-domain diagnostic figures to {FIG_DIR}')

## 8. Tertiary Multiclass Model

In [ ]:
multiclass_result = None
if config.fit_secondary_multiclass:
    y_multi_train = targets['multiclass'].loc[split.X_train.index]
    y_multi_test = targets['multiclass'].loc[split.X_test.index]
    multiclass_result = fit_multiclass_model(
        name='multiclass_cluster_type',
        X_train=split.X_train,
        y_train=y_multi_train,
        X_test=split.X_test,
        y_test=y_multi_test,
        config=config,
        numeric_features=feature_spec.numeric_features,
        categorical_features=feature_spec.categorical_features,
    )
    display(pd.DataFrame([multiclass_result.metrics]).round(3))
    display(multiclass_result.confusion_matrix)
    display(multiclass_result.classification_report.round(3))
if multiclass_result is not None:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(multiclass_result.confusion_matrix, annot=True, fmt=',d', cmap='Blues', ax=ax)
    ax.set_title('Multiclass cluster-type confusion matrix')
    fig.savefig(FIG_DIR / 'multiclass_cluster_type_confusion_matrix.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / 'multiclass_cluster_type_confusion_matrix.pdf', bbox_inches='tight')
    plt.show()
    print(f'Saved multiclass confusion matrix to {FIG_DIR}')

## 9. Sensitivity Analyses

In [ ]:
threshold_results = pd.DataFrame()
resolution_results = pd.DataFrame()

if config.run_threshold_sensitivity:
    threshold_results = run_threshold_sensitivity(raw, config)
    display(threshold_results)

if config.run_resolution_sensitivity:
    resolution_results = run_resolution_sensitivity(raw, config)
    display(resolution_results)

## 10. Direction Checks

In [ ]:
PDP_FEATURES = [
    'age_midpoint',
    feature_spec.simd_overall_feature,
    'is_female',
    'vacc_dose_number',
    'days_since_epoch',
    'days_since_vaccination_model',
    'is_reinfection',
    'dz_cum_prop_sequenced',
    'log_dz_cum_incidence_per_capita',
]
X_pdp = split.X_test.sample(n=min(20_000, len(split.X_test)), random_state=config.random_state)

pdp_curves = partial_dependence_curves(
    large_result.model,
    X_pdp,
    PDP_FEATURES,
    grid_resolution=30,
    positive_class_index=1,
)

pdp_curves.to_csv(OUT_DIR / 'binary_large_cluster_partial_dependence_curves.csv', index=False)

pdp_fig = plot_partial_dependence_curves(pdp_curves)
pdp_fig.savefig(FIG_DIR / 'binary_large_cluster_partial_dependence.png', dpi=300, bbox_inches='tight')
pdp_fig.savefig(FIG_DIR / 'binary_large_cluster_partial_dependence.pdf', bbox_inches='tight')

ALE_FEATURES = [
    'age_midpoint',
    feature_spec.simd_overall_feature,
    'vacc_dose_number',
    'days_since_epoch',
    'days_since_vaccination_model',
    'dz_cum_prop_sequenced',
    'log_dz_cum_incidence_per_capita',
    'log_wn_no_sequences',
]

ale_curves = accumulated_local_effects_curves(
    large_result.model,
    X_pdp,
    ALE_FEATURES,
    bins=20,
    positive_class_index=1,
)

ale_curves.to_csv(OUT_DIR / 'binary_large_cluster_ale_curves.csv', index=False)

ale_fig = plot_ale_curves(ale_curves)
ale_fig.savefig(FIG_DIR / 'binary_large_cluster_ale.png', dpi=300, bbox_inches='tight')
ale_fig.savefig(FIG_DIR / 'binary_large_cluster_ale.pdf', bbox_inches='tight')

if simd_domain_result is not None:
    X_decomp_pdp = X_decomp.loc[X_pdp.index].copy()
    SIMD_DOMAIN_ALE_FEATURES = feature_spec.simd_domain_features
    simd_domain_ale_curves = accumulated_local_effects_curves(
        simd_domain_result.model,
        X_decomp_pdp,
        SIMD_DOMAIN_ALE_FEATURES,
        bins=20,
        positive_class_index=1,
    )
    simd_domain_ale_curves.to_csv(
        OUT_DIR / 'simd_domain_large_cluster_ale_curves.csv',
        index=False,
    )

    simd_domain_ale_fig = plot_ale_curves(simd_domain_ale_curves)
    simd_domain_ale_fig.savefig(
        FIG_DIR / 'simd_domain_large_cluster_ale.png',
        dpi=300,
        bbox_inches='tight',
    )
    simd_domain_ale_fig.savefig(
        FIG_DIR / 'simd_domain_large_cluster_ale.pdf',
        bbox_inches='tight',
    )
    print(f'Saved SIMD-domain ALE curves to {OUT_DIR / "simd_domain_large_cluster_ale_curves.csv"}')

plt.show()

print(f'Saved PDP curves to {OUT_DIR / "binary_large_cluster_partial_dependence_curves.csv"}')
print(f'Saved ALE curves to {OUT_DIR / "binary_large_cluster_ale_curves.csv"}')
print(f'Saved PDP/ALE figures to {FIG_DIR}')
if simd_domain_result is not None:
    print(f'Saved SIMD-domain ALE figures to {FIG_DIR}')

## 11. Save Artefacts

In [ ]:
save_outcome_tables(outcome_tables, OUT_DIR)
save_binary_result(large_result, OUT_DIR, save_model=config.save_models)
if singleton_result is not None:
    save_binary_result(singleton_result, OUT_DIR, save_model=config.save_models)
if simd_domain_result is not None:
    save_binary_result(simd_domain_result, OUT_DIR, save_model=config.save_models)
if multiclass_result is not None:
    save_multiclass_result(multiclass_result, OUT_DIR, save_model=config.save_models)
save_sensitivity_metrics(threshold_results, resolution_results, OUT_DIR)
save_run_metadata(
    config,
    feature_spec,
    OUT_DIR,
    extra={
        'splitter': split.splitter_name,
        'sequence_membership_definition': 'One row per sequence; overlapping-window membership chosen by largest cluster size, then closest midpoint.',
    },
)
print(f'Saved outputs to {OUT_DIR}')

## 12. Optional SHAP

This is expensive with the final 1000-tree forest. Use a small sample first, and save the result immediately.

In [ ]:
RUN_SHAP = False
MAX_SHAP_ROWS = 1_000

if RUN_SHAP:
    from cluster_size_rf.lib.shap_utils import compute_tree_shap_for_positive_class, save_shap_artifacts

    X_shap_raw = split.X_test.sample(n=min(MAX_SHAP_ROWS, len(split.X_test)), random_state=config.random_state)
    large_shap, X_shap, feature_names = compute_tree_shap_for_positive_class(large_result.model, X_shap_raw)
    encoded_shap, grouped_shap = save_shap_artifacts(
        large_shap,
        X_shap,
        X_shap_raw,
        feature_names,
        feature_spec.all_features,
        OUT_DIR,
        prefix='binary_large_cluster',
    )
    display(grouped_shap.head(25))